# DS107 IAA Pilot Analysis

Notebook này phân tích chuyên sâu 2 vòng pilot topic annotation trong `data/04_Labeling_Pilot/`:

- Pilot set 100: phân tích toàn bộ 100 dòng.
- Pilot set 400: phân tích toàn bộ 400 dòng.
- Pilot set 400 theo batch: tách 4 batch liên tiếp, mỗi batch 100 dòng.

Các chỉ số chính gồm Fleiss' kappa cho 3 annotator, Cohen's kappa từng cặp, phân phối nhãn, nhãn chưa dùng, nhãn không phát sinh bất đồng, nhãn hay bất đồng, tần suất bất đồng và cặp nhãn hay bị nhầm/bất đồng với nhau.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from datetime import datetime
import math
import pandas as pd

# ---------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------
CWD = Path.cwd()
if (CWD / "data" / "04_Labeling_Pilot").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data" / "04_Labeling_Pilot").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Không tìm thấy data/04_Labeling_Pilot từ current directory hoặc parent directory.")

DATA_DIR = REPO_ROOT / "data" / "04_Labeling_Pilot"
PILOT100_PATH = DATA_DIR / "[DS107] Topic Annotation Pilot Set 100 - IAA_Merge.csv"
PILOT400_PATH = DATA_DIR / "[DS107] Topic Annotation Pilot Set 400 - IAA_Merge.csv"
REPORT_PATH = REPO_ROOT / "iaa_analysis_techinical_report.md"

ANNOTATOR_COLS = {
    "Nhung": "annotator_Nhung_topic_label",
    "Yen": "annotator_Yen_topic_label",
    "Han": "annotator_Han_topic_label",
}
ANNOTATORS = list(ANNOTATOR_COLS.keys())
AGREEMENT_ORDER = ["full_agreement", "partial_disagreement", "complete_disagreement", "incomplete"]


def normalize_label(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None


def fmt_pct(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "NA"
    return f"{value * 100:.1f}%"


def fmt_num(value, digits=4):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "NA"
    return f"{value:.{digits}f}"


def clean_cell(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = str(value).replace("\n", " ").replace("|", "\\|")
    return text


def md_table(rows, headers):
    if isinstance(rows, pd.DataFrame):
        headers = list(rows.columns)
        rows = rows.to_dict("records")
    if not rows:
        return "_Không có._"
    if isinstance(rows[0], dict):
        body = [[clean_cell(row.get(h, "")) for h in headers] for row in rows]
    else:
        body = [[clean_cell(v) for v in row] for row in rows]
    header = "| " + " | ".join(clean_cell(h) for h in headers) + " |"
    sep = "| " + " | ".join("---" for _ in headers) + " |"
    lines = [header, sep]
    lines += ["| " + " | ".join(row) + " |" for row in body]
    return "\n".join(lines)


def read_pilot(path):
    df = pd.read_csv(path)
    missing_cols = [col for col in ANNOTATOR_COLS.values() if col not in df.columns]
    if missing_cols:
        raise ValueError(f"File {path.name} thiếu cột annotator: {missing_cols}")
    df = df.copy()
    df["source_file"] = path.name
    df["row_in_file"] = range(1, len(df) + 1)
    return df


def extract_label_frame(df):
    labels = df[list(ANNOTATOR_COLS.values())].copy()
    labels.columns = ANNOTATORS
    for col in labels.columns:
        labels[col] = labels[col].map(normalize_label)
    return labels


def build_label_universe(*dfs):
    labels = set()
    label_to_id = {}
    final_label_cols = ["topic_label_final", "topic_label_final_auto", "topic_label_final_train"]
    final_id_cols = ["topic_label_final_id", "topic_label_final_id_train"]

    for df in dfs:
        for col in list(ANNOTATOR_COLS.values()) + final_label_cols:
            if col in df.columns:
                labels.update(v for v in df[col].map(normalize_label).dropna().tolist())

        for label_col, id_col in [("topic_label_final", "topic_label_final_id"), ("topic_label_final_train", "topic_label_final_id_train")]:
            if label_col in df.columns and id_col in df.columns:
                pairs = df[[label_col, id_col]].dropna()
                for label, label_id in pairs.itertuples(index=False):
                    label = normalize_label(label)
                    label_id = normalize_label(label_id)
                    if label and label_id and label not in label_to_id:
                        label_to_id[label] = label_id

    ordered = sorted(labels, key=lambda label: (label_to_id.get(label, "T99"), label))
    return ordered, label_to_id


def cohen_kappa(labels_a, labels_b, label_universe):
    pairs = [(a, b) for a, b in zip(labels_a, labels_b) if a is not None and b is not None]
    n = len(pairs)
    if n == 0:
        return {"n": 0, "observed_agreement": math.nan, "expected_agreement": math.nan, "kappa": math.nan}
    agree = sum(1 for a, b in pairs if a == b)
    observed = agree / n
    counts_a = Counter(a for a, _ in pairs)
    counts_b = Counter(b for _, b in pairs)
    expected = sum((counts_a[label] / n) * (counts_b[label] / n) for label in label_universe)
    denom = 1 - expected
    kappa = 1.0 if denom == 0 and observed == 1 else ((observed - expected) / denom if denom != 0 else math.nan)
    return {"n": n, "observed_agreement": observed, "expected_agreement": expected, "kappa": kappa}


def fleiss_kappa(label_df, label_universe):
    valid = label_df.dropna()
    n_items = len(valid)
    n_raters = len(label_df.columns)
    if n_items == 0 or n_raters < 2:
        return {"n_items": n_items, "n_raters": n_raters, "observed_agreement": math.nan, "expected_agreement": math.nan, "kappa": math.nan}

    row_agreements = []
    label_totals = Counter()
    for row in valid.itertuples(index=False):
        counts = Counter(row)
        label_totals.update(counts)
        row_agreement = sum(count * (count - 1) for count in counts.values()) / (n_raters * (n_raters - 1))
        row_agreements.append(row_agreement)

    observed = sum(row_agreements) / n_items
    total_assignments = n_items * n_raters
    expected = sum((label_totals[label] / total_assignments) ** 2 for label in label_universe)
    denom = 1 - expected
    kappa = 1.0 if denom == 0 and observed == 1 else ((observed - expected) / denom if denom != 0 else math.nan)
    return {"n_items": n_items, "n_raters": n_raters, "observed_agreement": observed, "expected_agreement": expected, "kappa": kappa}


def agreement_type(row):
    values = [v for v in row if v is not None]
    if len(values) < len(ANNOTATORS):
        return "incomplete"
    unique_count = len(set(values))
    if unique_count == 1:
        return "full_agreement"
    if unique_count == 2:
        return "partial_disagreement"
    return "complete_disagreement"


def majority_label(row):
    values = [v for v in row if v is not None]
    if len(values) < len(ANNOTATORS):
        return None
    counts = Counter(values)
    top_count = max(counts.values())
    if top_count >= 2:
        return sorted([label for label, count in counts.items() if count == top_count])[0]
    return "NO_MAJORITY_TIE"


def label_distribution(label_df, label_universe):
    rows = []
    for label in label_universe:
        row = {"label": label}
        row["total_assignments"] = int((label_df == label).sum().sum())
        for annotator in ANNOTATORS:
            row[annotator] = int((label_df[annotator] == label).sum())
        rows.append(row)
    out = pd.DataFrame(rows)
    total = out["total_assignments"].sum()
    out["assignment_pct"] = out["total_assignments"].map(lambda n: n / total if total else 0)
    return out.sort_values(["total_assignments", "label"], ascending=[False, True]).reset_index(drop=True)


def annotator_alignment_table(label_df, label_universe):
    valid = label_df.dropna().copy()
    rows = []
    total_assignments_by_annotator = {annotator: int(label_df[annotator].notna().sum()) for annotator in ANNOTATORS}
    annotator_distributions = {}
    for annotator in ANNOTATORS:
        denom = total_assignments_by_annotator[annotator]
        counts = label_df[annotator].value_counts()
        annotator_distributions[annotator] = {label: counts.get(label, 0) / denom if denom else 0 for label in label_universe}
    group_mean_distribution = {
        label: sum(annotator_distributions[annotator][label] for annotator in ANNOTATORS) / len(ANNOTATORS)
        for label in label_universe
    }

    for annotator in ANNOTATORS:
        others = [other for other in ANNOTATORS if other != annotator]
        pair_kappas = []
        pair_agreements = []
        pair_disagreement_rates = []
        for other in others:
            metric = cohen_kappa(label_df[annotator].tolist(), label_df[other].tolist(), label_universe)
            pair_kappas.append(metric["kappa"])
            pair_agreements.append(metric["observed_agreement"])
            pair_disagreement_rates.append(1 - metric["observed_agreement"] if not math.isnan(metric["observed_agreement"]) else math.nan)

        if len(valid):
            other_consensus_mask = valid[others[0]] == valid[others[1]]
            n_other_consensus = int(other_consensus_mask.sum())
            n_match_other_consensus = int((valid.loc[other_consensus_mask, annotator] == valid.loc[other_consensus_mask, others[0]]).sum())
            n_minority_when_other2_agree = n_other_consensus - n_match_other_consensus
        else:
            n_other_consensus = 0
            n_match_other_consensus = 0
            n_minority_when_other2_agree = 0

        l1_to_group_mean = sum(abs(annotator_distributions[annotator][label] - group_mean_distribution[label]) for label in label_universe)
        rows.append({
            "annotator": annotator,
            "avg_pairwise_cohen": sum(pair_kappas) / len(pair_kappas) if pair_kappas else math.nan,
            "avg_pairwise_agreement": sum(pair_agreements) / len(pair_agreements) if pair_agreements else math.nan,
            "avg_pairwise_disagreement_rate": sum(pair_disagreement_rates) / len(pair_disagreement_rates) if pair_disagreement_rates else math.nan,
            "other2_consensus_rows": n_other_consensus,
            "match_other2_consensus_rows": n_match_other_consensus,
            "match_other2_consensus_rate": n_match_other_consensus / n_other_consensus if n_other_consensus else math.nan,
            "minority_when_other2_agree_rows": n_minority_when_other2_agree,
            "minority_when_other2_agree_rate": n_minority_when_other2_agree / n_other_consensus if n_other_consensus else math.nan,
            "label_distribution_l1_to_group_mean": l1_to_group_mean,
        })
    out = pd.DataFrame(rows)
    out["average_similarity_rank"] = out.sort_values(
        ["avg_pairwise_cohen", "match_other2_consensus_rate", "label_distribution_l1_to_group_mean"],
        ascending=[False, False, True],
    ).reset_index(drop=True).index + 1
    rank_map = out.sort_values(
        ["avg_pairwise_cohen", "match_other2_consensus_rate", "label_distribution_l1_to_group_mean"],
        ascending=[False, False, True],
    )["annotator"].reset_index(drop=True).reset_index().set_index("annotator")["index"].map(lambda i: i + 1)
    out["average_similarity_rank"] = out["annotator"].map(rank_map)
    out["outlier_rank"] = out.sort_values(
        ["avg_pairwise_cohen", "match_other2_consensus_rate", "label_distribution_l1_to_group_mean"],
        ascending=[True, True, False],
    )["annotator"].reset_index(drop=True).reset_index().set_index("annotator")["index"].map(lambda i: i + 1).reindex(out["annotator"]).to_numpy()
    return out.sort_values("average_similarity_rank").reset_index(drop=True)


def analyze_dataset(name, df, label_universe, label_to_id, row_scope):
    label_df = extract_label_frame(df)
    valid_mask = label_df.notna().all(axis=1)
    valid = label_df[valid_mask].copy()
    types = label_df.apply(lambda row: agreement_type(row.tolist()), axis=1)
    valid_types = types[valid_mask]

    fleiss = fleiss_kappa(label_df, label_universe)
    cohen_rows = []
    for a, b in combinations(ANNOTATORS, 2):
        metric = cohen_kappa(label_df[a].tolist(), label_df[b].tolist(), label_universe)
        cohen_rows.append({
            "pair": f"{a} vs {b}",
            "n": metric["n"],
            "observed_agreement": metric["observed_agreement"],
            "expected_agreement": metric["expected_agreement"],
            "cohen_kappa": metric["kappa"],
        })
    cohen_df = pd.DataFrame(cohen_rows)

    type_counts = types.value_counts().reindex(AGREEMENT_ORDER, fill_value=0).reset_index()
    type_counts.columns = ["agreement_type", "rows"]
    type_counts["pct_all_rows"] = type_counts["rows"].map(lambda n: n / len(df) if len(df) else 0)

    valid_disagreement_mask = valid_types.isin(["partial_disagreement", "complete_disagreement"])
    disagreement_rows = valid[valid_disagreement_mask].copy()
    n_disagreement_rows = len(disagreement_rows)

    used_labels = set(v for col in label_df.columns for v in label_df[col].dropna().tolist() if v is not None)
    unused_labels = [label for label in label_universe if label not in used_labels]
    dist = label_distribution(label_df, label_universe)

    majority = valid.apply(lambda row: majority_label(row.tolist()), axis=1)
    majority_counts = majority.value_counts().rename_axis("label").reset_index(name="rows")
    majority_counts["pct_valid_rows"] = majority_counts["rows"].map(lambda n: n / len(valid) if len(valid) else 0)

    label_rows = []
    for label in label_universe:
        rows_with_label = valid.apply(lambda row: label in set(row.tolist()), axis=1) if len(valid) else pd.Series(dtype=bool)
        rows_with_label_count = int(rows_with_label.sum()) if len(valid) else 0
        disagreement_with_label = int((rows_with_label & valid_disagreement_mask).sum()) if len(valid) else 0
        unanimous_for_label = int((valid == label).all(axis=1).sum()) if len(valid) else 0
        assignment_count = int((valid == label).sum().sum()) if len(valid) else 0
        disagreement_assignment_count = int((disagreement_rows == label).sum().sum()) if n_disagreement_rows else 0
        label_rows.append({
            "label": label,
            "label_id": label_to_id.get(label, ""),
            "assignment_count_valid": assignment_count,
            "rows_with_label": rows_with_label_count,
            "unanimous_rows": unanimous_for_label,
            "disagreement_rows_with_label": disagreement_with_label,
            "disagreement_assignment_count": disagreement_assignment_count,
            "disagreement_rate_when_label_appears": disagreement_with_label / rows_with_label_count if rows_with_label_count else math.nan,
        })
    label_stats = pd.DataFrame(label_rows)
    no_disagreement_labels = label_stats[(label_stats["rows_with_label"] > 0) & (label_stats["disagreement_rows_with_label"] == 0)].copy()
    frequent_disagreement_labels = label_stats[label_stats["disagreement_rows_with_label"] > 0].sort_values(
        ["disagreement_rows_with_label", "disagreement_rate_when_label_appears", "assignment_count_valid", "label"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    confusion_counter = Counter()
    for row in disagreement_rows.itertuples(index=False):
        distinct_labels = sorted(set(row))
        for a, b in combinations(distinct_labels, 2):
            confusion_counter[(a, b)] += 1
    confusion_rows = []
    for (a, b), count in confusion_counter.most_common():
        confusion_rows.append({
            "label_a": a,
            "label_b": b,
            "disagreement_rows": count,
            "pct_of_disagreement_rows": count / n_disagreement_rows if n_disagreement_rows else 0,
        })
    confusion_df = pd.DataFrame(confusion_rows)

    pairwise_rows = []
    for a, b in combinations(ANNOTATORS, 2):
        pair_df = label_df[[a, b]].dropna()
        n = len(pair_df)
        disagreements = int((pair_df[a] != pair_df[b]).sum()) if n else 0
        pairwise_rows.append({
            "pair": f"{a} vs {b}",
            "n": n,
            "agreements": n - disagreements,
            "disagreements": disagreements,
            "disagreement_rate": disagreements / n if n else math.nan,
        })
    pairwise_disagreement_df = pd.DataFrame(pairwise_rows)

    return {
        "name": name,
        "row_scope": row_scope,
        "rows_total": len(df),
        "rows_valid_3_annotators": int(valid_mask.sum()),
        "rows_incomplete": int((~valid_mask).sum()),
        "labels_used_count": len(used_labels),
        "labels_unused_count": len(unused_labels),
        "unused_labels": unused_labels,
        "fleiss": fleiss,
        "cohen": cohen_df,
        "agreement_counts": type_counts,
        "assignment_distribution": dist,
        "majority_distribution": majority_counts,
        "annotator_alignment": annotator_alignment_table(label_df, label_universe),
        "label_stats": label_stats,
        "no_disagreement_labels": no_disagreement_labels.sort_values(["label_id", "label"]).reset_index(drop=True),
        "frequent_disagreement_labels": frequent_disagreement_labels,
        "confusion_pairs": confusion_df,
        "pairwise_disagreements": pairwise_disagreement_df,
    }


def format_metric_tables(result):
    cohen = result["cohen"].copy()
    for col in ["observed_agreement", "expected_agreement", "cohen_kappa"]:
        cohen[col] = cohen[col].map(fmt_num)
    fleiss = result["fleiss"]
    fleiss_rows = [{
        "n_items": fleiss["n_items"],
        "n_raters": fleiss["n_raters"],
        "observed_agreement": fmt_num(fleiss["observed_agreement"]),
        "expected_agreement": fmt_num(fleiss["expected_agreement"]),
        "fleiss_kappa": fmt_num(fleiss["kappa"]),
    }]
    return fleiss_rows, cohen


def format_annotator_alignment(result):
    alignment = result["annotator_alignment"].copy()
    for col in ["avg_pairwise_cohen", "label_distribution_l1_to_group_mean"]:
        alignment[col] = alignment[col].map(fmt_num)
    for col in ["avg_pairwise_agreement", "avg_pairwise_disagreement_rate", "match_other2_consensus_rate", "minority_when_other2_agree_rate"]:
        alignment[col] = alignment[col].map(fmt_pct)
    return alignment[[
        "average_similarity_rank",
        "outlier_rank",
        "annotator",
        "avg_pairwise_cohen",
        "avg_pairwise_agreement",
        "avg_pairwise_disagreement_rate",
        "other2_consensus_rows",
        "match_other2_consensus_rows",
        "match_other2_consensus_rate",
        "minority_when_other2_agree_rows",
        "minority_when_other2_agree_rate",
        "label_distribution_l1_to_group_mean",
    ]]


def print_analysis(result, top_n=12):
    print("=" * 96)
    print(result["name"])
    print("=" * 96)
    print(f"Phạm vi dòng: {result['row_scope']}")
    print(f"Tổng dòng: {result['rows_total']} | Dòng đủ 3 nhãn: {result['rows_valid_3_annotators']} | Dòng thiếu nhãn: {result['rows_incomplete']}")
    print(f"Nhãn được dùng: {result['labels_used_count']} | Nhãn không được dùng: {result['labels_unused_count']}")
    if result["unused_labels"]:
        print("Nhãn không được dùng:", ", ".join(result["unused_labels"]))
    print("\nFleiss' kappa")
    fleiss_rows, cohen = format_metric_tables(result)
    print(md_table(fleiss_rows, list(fleiss_rows[0].keys())))
    print("\nCohen's kappa từng cặp")
    print(md_table(cohen, list(cohen.columns)))
    alignment = format_annotator_alignment(result)
    print("\nAnnotator giống trung bình nhóm / khác biệt so với nhóm")
    print(md_table(alignment, list(alignment.columns)))

    agreement = result["agreement_counts"].copy()
    agreement["pct_all_rows"] = agreement["pct_all_rows"].map(fmt_pct)
    print("\nTần suất đồng thuận/bất đồng")
    print(md_table(agreement, list(agreement.columns)))

    dist = result["assignment_distribution"].copy()
    dist["assignment_pct"] = dist["assignment_pct"].map(fmt_pct)
    print("\nPhân phối nhãn theo toàn bộ lượt gán")
    print(md_table(dist, list(dist.columns)))

    clean = result["no_disagreement_labels"][["label_id", "label", "rows_with_label", "assignment_count_valid"]]
    print("\nNhãn không hề phát sinh bất đồng khi xuất hiện")
    print(md_table(clean, list(clean.columns)))

    frequent = result["frequent_disagreement_labels"][[
        "label_id", "label", "rows_with_label", "disagreement_rows_with_label", "disagreement_rate_when_label_appears", "disagreement_assignment_count"
    ]].head(top_n).copy()
    frequent["disagreement_rate_when_label_appears"] = frequent["disagreement_rate_when_label_appears"].map(fmt_pct)
    print("\nNhãn hay phát sinh bất đồng")
    print(md_table(frequent, list(frequent.columns)))

    pairs = result["confusion_pairs"].head(top_n).copy()
    if not pairs.empty:
        pairs["pct_of_disagreement_rows"] = pairs["pct_of_disagreement_rows"].map(fmt_pct)
    print("\nCặp nhãn hay bị bất đồng với nhau")
    print(md_table(pairs, list(pairs.columns) if not pairs.empty else ["label_a", "label_b", "disagreement_rows", "pct_of_disagreement_rows"]))


def overview_table(results):
    rows = []
    for result in results:
        cohen_mean = result["cohen"]["cohen_kappa"].mean()
        counts = dict(zip(result["agreement_counts"]["agreement_type"], result["agreement_counts"]["rows"]))
        rows.append({
            "dataset": result["name"],
            "rows": result["rows_total"],
            "valid_rows": result["rows_valid_3_annotators"],
            "incomplete": result["rows_incomplete"],
            "fleiss_kappa": fmt_num(result["fleiss"]["kappa"]),
            "mean_cohen_kappa": fmt_num(cohen_mean),
            "used_labels": result["labels_used_count"],
            "unused_labels": result["labels_unused_count"],
            "full": counts.get("full_agreement", 0),
            "partial": counts.get("partial_disagreement", 0),
            "complete": counts.get("complete_disagreement", 0),
        })
    return pd.DataFrame(rows)


def result_to_markdown(result, top_n=15):
    fleiss_rows, cohen = format_metric_tables(result)
    alignment = format_annotator_alignment(result)
    agreement = result["agreement_counts"].copy()
    agreement["pct_all_rows"] = agreement["pct_all_rows"].map(fmt_pct)
    dist = result["assignment_distribution"].copy()
    dist["assignment_pct"] = dist["assignment_pct"].map(fmt_pct)
    clean = result["no_disagreement_labels"][["label_id", "label", "rows_with_label", "assignment_count_valid"]]
    frequent = result["frequent_disagreement_labels"][[
        "label_id", "label", "rows_with_label", "disagreement_rows_with_label", "disagreement_rate_when_label_appears", "disagreement_assignment_count"
    ]].head(top_n).copy()
    frequent["disagreement_rate_when_label_appears"] = frequent["disagreement_rate_when_label_appears"].map(fmt_pct)
    pairs = result["confusion_pairs"].head(top_n).copy()
    if not pairs.empty:
        pairs["pct_of_disagreement_rows"] = pairs["pct_of_disagreement_rows"].map(fmt_pct)

    sections = [
        f"## {result['name']}",
        f"Phạm vi dòng: {result['row_scope']}. Tổng dòng: {result['rows_total']}; dòng đủ 3 nhãn: {result['rows_valid_3_annotators']}; dòng thiếu nhãn: {result['rows_incomplete']}.",
        f"Nhãn được dùng: {result['labels_used_count']}/{len(LABEL_UNIVERSE)}. Nhãn không được dùng: {result['labels_unused_count']}" + (f" ({', '.join(result['unused_labels'])})." if result['unused_labels'] else "."),
        "### Fleiss' kappa",
        md_table(fleiss_rows, list(fleiss_rows[0].keys())),
        "### Cohen's kappa từng cặp",
        md_table(cohen, list(cohen.columns)),
        "### Annotator giống trung bình nhóm / khác biệt so với nhóm",
        md_table(alignment, list(alignment.columns)),
        "### Tần suất đồng thuận/bất đồng",
        md_table(agreement, list(agreement.columns)),
        "### Phân phối nhãn theo lượt gán",
        md_table(dist, list(dist.columns)),
        "### Nhãn không hề phát sinh bất đồng khi xuất hiện",
        md_table(clean, list(clean.columns)),
        "### Nhãn hay phát sinh bất đồng",
        md_table(frequent, list(frequent.columns)),
        "### Cặp nhãn hay bị bất đồng với nhau",
        md_table(pairs, list(pairs.columns) if not pairs.empty else ["label_a", "label_b", "disagreement_rows", "pct_of_disagreement_rows"]),
    ]
    return "\n\n".join(sections)


def build_report(results_full, batch_results, combined_result=None):
    all_results = ([combined_result] if combined_result is not None else []) + results_full + batch_results
    overview = overview_table(all_results)
    label_rows = [{"label_id": LABEL_TO_ID.get(label, ""), "label": label} for label in LABEL_UNIVERSE]

    r100, r400 = results_full
    fleiss_delta = r400["fleiss"]["kappa"] - r100["fleiss"]["kappa"]
    disagreement_100 = (r100["agreement_counts"].set_index("agreement_type").loc[["partial_disagreement", "complete_disagreement"], "rows"].sum() / r100["rows_total"])
    disagreement_400 = (r400["agreement_counts"].set_index("agreement_type").loc[["partial_disagreement", "complete_disagreement"], "rows"].sum() / r400["rows_total"])
    worst_batch = min(batch_results, key=lambda r: r["fleiss"]["kappa"])
    best_batch = max(batch_results, key=lambda r: r["fleiss"]["kappa"])
    top_pair_400 = r400["confusion_pairs"].iloc[0].to_dict() if not r400["confusion_pairs"].empty else None
    top_label_400 = r400["frequent_disagreement_labels"].iloc[0].to_dict() if not r400["frequent_disagreement_labels"].empty else None
    annotator_overview_rows = []
    for result in all_results:
        alignment = result["annotator_alignment"]
        most_average = alignment.sort_values("average_similarity_rank").iloc[0]
        outlier = alignment.sort_values("outlier_rank").iloc[0]
        annotator_overview_rows.append({
            "dataset": result["name"],
            "most_average_annotator": most_average["annotator"],
            "most_average_avg_cohen": fmt_num(most_average["avg_pairwise_cohen"]),
            "most_average_match_other2_consensus": fmt_pct(most_average["match_other2_consensus_rate"]),
            "outlier_annotator": outlier["annotator"],
            "outlier_avg_cohen": fmt_num(outlier["avg_pairwise_cohen"]),
            "outlier_minority_when_other2_agree": fmt_pct(outlier["minority_when_other2_agree_rate"]),
            "outlier_label_dist_l1": fmt_num(outlier["label_distribution_l1_to_group_mean"]),
        })
    annotator_overview = pd.DataFrame(annotator_overview_rows)
    r400_alignment = r400["annotator_alignment"]
    r400_most_average = r400_alignment.sort_values("average_similarity_rank").iloc[0]
    r400_outlier = r400_alignment.sort_values("outlier_rank").iloc[0]
    if combined_result is not None:
        combined_majority_distribution = combined_result["majority_distribution"].copy()
        combined_majority_distribution["pct_all_500_rows"] = combined_majority_distribution["rows"].map(lambda n: n / combined_result["rows_total"] if combined_result["rows_total"] else 0)
        if combined_result["rows_incomplete"]:
            combined_majority_distribution = pd.concat([
                combined_majority_distribution,
                pd.DataFrame([{"label": "INCOMPLETE_MISSING_LABELS", "rows": combined_result["rows_incomplete"], "pct_valid_rows": math.nan, "pct_all_500_rows": combined_result["rows_incomplete"] / combined_result["rows_total"]}]),
            ], ignore_index=True)
        combined_majority_distribution = combined_majority_distribution.sort_values(["rows", "label"], ascending=[False, True]).reset_index(drop=True)
        combined_majority_distribution["pct_valid_rows"] = combined_majority_distribution["pct_valid_rows"].map(fmt_pct)
        combined_majority_distribution["pct_all_500_rows"] = combined_majority_distribution["pct_all_500_rows"].map(fmt_pct)

        combined_assignment_distribution = combined_result["assignment_distribution"].copy()
        combined_assignment_distribution["assignment_pct"] = combined_assignment_distribution["assignment_pct"].map(fmt_pct)

        combined_disagreement_labels = combined_result["frequent_disagreement_labels"][[
            "label_id", "label", "rows_with_label", "disagreement_rows_with_label", "disagreement_rate_when_label_appears", "disagreement_assignment_count"
        ]].copy()
        combined_disagreement_labels["disagreement_rate_when_label_appears"] = combined_disagreement_labels["disagreement_rate_when_label_appears"].map(fmt_pct)

        combined_disagreement_pairs = combined_result["confusion_pairs"].head(20).copy()
        if not combined_disagreement_pairs.empty:
            combined_disagreement_pairs["pct_of_disagreement_rows"] = combined_disagreement_pairs["pct_of_disagreement_rows"].map(fmt_pct)
    else:
        combined_majority_distribution = pd.DataFrame()
        combined_assignment_distribution = pd.DataFrame()
        combined_disagreement_labels = pd.DataFrame()
        combined_disagreement_pairs = pd.DataFrame()

    insight_lines = [
        f"- Fleiss' kappa tăng từ {fmt_num(r100['fleiss']['kappa'])} ở pilot 100 lên {fmt_num(r400['fleiss']['kappa'])} ở pilot 400 (delta {fmt_num(fleiss_delta)}), cho thấy mức nhất quán 3 người tốt hơn ở vòng 2.",
        f"- Tỷ lệ dòng có bất đồng giảm từ {fmt_pct(disagreement_100)} ở pilot 100 xuống {fmt_pct(disagreement_400)} ở pilot 400. Pilot 400 còn 3 dòng thiếu nhãn của Nhung/Yen nên bị loại khỏi phép tính IAA 3 người.",
        f"- Batch ổn định nhất trong file 400 là {best_batch['name']} với Fleiss {fmt_num(best_batch['fleiss']['kappa'])}; batch yếu nhất là {worst_batch['name']} với Fleiss {fmt_num(worst_batch['fleiss']['kappa'])}.",
    ]
    if top_label_400:
        insight_lines.append(f"- Trong pilot 400, nhãn phát sinh bất đồng nhiều nhất là {top_label_400['label']} ({int(top_label_400['disagreement_rows_with_label'])} dòng có bất đồng khi nhãn này xuất hiện).")
    if top_pair_400:
        insight_lines.append(f"- Cặp nhãn hay bị đặt cạnh nhau trong các dòng bất đồng nhất của pilot 400 là {top_pair_400['label_a']} vs {top_pair_400['label_b']} ({int(top_pair_400['disagreement_rows'])} dòng).")
    if r400["no_disagreement_labels"].empty:
        insight_lines.append("- Ở pilot 400 không có nhãn nào vừa được dùng vừa hoàn toàn không phát sinh bất đồng; điều này hợp lý vì tập lớn hơn phủ đủ 18 nhãn và có nhiều ngữ cảnh giáp ranh.")
    else:
        clean_labels = ", ".join(r400["no_disagreement_labels"]["label"].tolist())
        insight_lines.append(f"- Ở pilot 400, các nhãn được dùng nhưng không phát sinh bất đồng là: {clean_labels}.")
    if combined_result is not None:
        top_majority_500 = ", ".join(f"{row.label} ({int(row.rows)} dòng)" for row in combined_result["majority_distribution"].head(3).itertuples(index=False))
        top_disagreement_labels_500 = ", ".join(f"{row.label} ({int(row.disagreement_rows_with_label)} dòng)" for row in combined_result["frequent_disagreement_labels"].head(3).itertuples(index=False))
        if not combined_result["confusion_pairs"].empty:
            top_pair_500 = combined_result["confusion_pairs"].iloc[0]
            top_pair_text_500 = f"{top_pair_500['label_a']} vs {top_pair_500['label_b']} ({int(top_pair_500['disagreement_rows'])} dòng)"
        else:
            top_pair_text_500 = "không có cặp bất đồng"
        insight_lines.append(f"- Trong 500 dòng gộp, các nhãn majority phổ biến nhất là {top_majority_500}; các nhãn phát sinh bất đồng nhiều nhất là {top_disagreement_labels_500}; cặp bất đồng nhiều nhất là {top_pair_text_500}.")
    insight_lines.append(f"- Về annotator, trong pilot 400 người giống trung bình nhóm nhất là {r400_most_average['annotator']} (Cohen trung bình {fmt_num(r400_most_average['avg_pairwise_cohen'])}, khớp majority của 2 người còn lại {fmt_pct(r400_most_average['match_other2_consensus_rate'])}); người khác biệt nhất là {r400_outlier['annotator']} (Cohen trung bình {fmt_num(r400_outlier['avg_pairwise_cohen'])}, lệch khỏi majority của 2 người còn lại {fmt_pct(r400_outlier['minority_when_other2_agree_rate'])}).")

    annotator_method_lines = [
        "- `most_average_annotator` được xếp theo Cohen's kappa trung bình với 2 annotator còn lại, sau đó xét tỷ lệ khớp với majority của 2 người còn lại và độ lệch phân phối nhãn so với trung bình nhóm.",
        "- `outlier_annotator` là chiều ngược lại: Cohen trung bình thấp hơn, ít khớp với majority của 2 người còn lại hơn, hoặc có phân phối nhãn lệch hơn.",
        "- `match_other2_consensus` chỉ tính trên các dòng mà 2 annotator còn lại đồng thuận; đây là thước đo trực quan xem annotator đó có hay là người thứ ba khác ý kiến không.",
        "- `label_dist_l1` càng thấp thì phân phối nhãn của annotator càng gần phân phối trung bình nhóm; chỉ số này hỗ trợ phát hiện bias dùng nhãn, không thay thế Cohen/majority agreement.",
    ]

    guideline_lines = [
        "- Gợi ý ưu tiên chỉnh sửa các nhãn/cụm nhãn gây bất đồng nhiều nhất trong pilot 400: `HUMAN_INTEREST`, `ARTS_CULTURE_ENTERTAINMENT_AND_MEDIA`, `LIFESTYLE_AND_LEISURE`, `OTHER_UNCLEAR`, `ECONOMY_BUSINESS_AND_FINANCE`, `CRIME_LAW_AND_JUSTICE`, `SOCIETY`.",
        "- Mỗi nhãn nên có 4 phần bắt buộc: định nghĩa ngắn, điều kiện include, điều kiện exclude, và quy tắc ưu tiên khi bài viết chạm nhiều chủ đề.",
        "- Với các nhãn có tỷ lệ bất đồng cao khi xuất hiện như `LIFESTYLE_AND_LEISURE` (90.9%), `SOCIETY` (77.8%), `HUMAN_INTEREST` (58.5%) và `OTHER_UNCLEAR` (58.1%), phiên bản guideline tiếp theo nên bổ sung ví dụ dương/âm cụ thể thay vì chỉ mô tả khái niệm.",
        "- `OTHER_UNCLEAR` cần được định nghĩa như nhãn fallback có điều kiện: chỉ dùng khi thiếu ngữ cảnh để xác định topic chính, không dùng chỉ vì nội dung ngắn, hài hước, bình luận hoặc có văn phong mạng xã hội.",
        "- Nên thêm decision tree cho annotator: xác định topic chính trước, sau đó kiểm tra các cặp ranh giới dễ nhầm, cuối cùng mới dùng `OTHER_UNCLEAR` nếu không đủ bằng chứng.",
        "- Nên tạo một phụ lục guideline riêng cho các case giáp ranh, lấy trực tiếp từ những dòng bất đồng trong pilot 400 và ghi rõ nhãn đúng sau adjudication cùng lý do chọn nhãn.",
        "- Sau khi cập nhật guideline, nên chạy một mini-calibration 50-100 dòng tập trung vào các cặp nhãn dưới đây trước khi bước sang batch gán nhãn lớn hơn.",
    ]
    guideline_boundary_rows = [
        {"suggested_boundary": "ARTS_CULTURE_ENTERTAINMENT_AND_MEDIA vs HUMAN_INTEREST", "pilot400_rows": 17, "suggested_revision": "Làm rõ khi nào bài về nhân vật/câu chuyện đời sống của người nổi tiếng vẫn là ARTS/MEDIA, và khi nào trọng tâm chuyển sang câu chuyện con người/cảm xúc nên là HUMAN_INTEREST."},
        {"suggested_boundary": "HUMAN_INTEREST vs LIFESTYLE_AND_LEISURE", "pilot400_rows": 9, "suggested_revision": "Tách nội dung trải nghiệm/câu chuyện cá nhân khỏi nội dung hướng dẫn thói quen, tiêu dùng, giải trí, du lịch, ăn uống hoặc phong cách sống."},
        {"suggested_boundary": "ARTS_CULTURE_ENTERTAINMENT_AND_MEDIA vs LIFESTYLE_AND_LEISURE", "pilot400_rows": 8, "suggested_revision": "Quy định ưu tiên ARTS/MEDIA khi trọng tâm là sản phẩm/hoạt động giải trí, văn hóa, showbiz, truyền thông; ưu tiên LIFESTYLE khi trọng tâm là hành vi sống, leisure hoặc lựa chọn cá nhân."},
        {"suggested_boundary": "LIFESTYLE_AND_LEISURE vs OTHER_UNCLEAR", "pilot400_rows": 7, "suggested_revision": "Nêu rõ nội dung ngắn về ăn chơi, du lịch, thời trang, giải trí cá nhân vẫn có thể là LIFESTYLE nếu có tín hiệu chủ đề đủ rõ; không đẩy sang OTHER_UNCLEAR chỉ vì thiếu văn cảnh rộng."},
        {"suggested_boundary": "ARTS_CULTURE_ENTERTAINMENT_AND_MEDIA vs OTHER_UNCLEAR", "pilot400_rows": 6, "suggested_revision": "Bổ sung tiêu chí nhận diện nội dung media/giải trí từ tên người, chương trình, phim, nhạc, sự kiện văn hóa; OTHER_UNCLEAR chỉ khi không xác định được thực thể hoặc sự kiện chính."},
        {"suggested_boundary": "HUMAN_INTEREST vs OTHER_UNCLEAR", "pilot400_rows": 5, "suggested_revision": "Nếu bài có câu chuyện con người, cảm xúc, hoàn cảnh cá nhân/cộng đồng đủ nhận diện thì ưu tiên HUMAN_INTEREST; nếu chỉ là caption/bình luận không có sự kiện hoặc chủ thể rõ thì mới OTHER_UNCLEAR."},
        {"suggested_boundary": "HUMAN_INTEREST vs SOCIETY", "pilot400_rows": 5, "suggested_revision": "Tách câu chuyện con người đơn lẻ khỏi vấn đề xã hội có phạm vi cộng đồng, chính sách, phúc lợi, dịch vụ công hoặc tác động xã hội rộng."},
        {"suggested_boundary": "CRIME_LAW_AND_JUSTICE vs POLITICS", "pilot400_rows": 5, "suggested_revision": "Ưu tiên CRIME/LAW khi trọng tâm là điều tra, xét xử, vi phạm, bắt giữ, án phạt; ưu tiên POLITICS khi trọng tâm là hoạt động nhà nước, đảng, bầu cử, ngoại giao hoặc quyết sách."},
        {"suggested_boundary": "ECONOMY_BUSINESS_AND_FINANCE vs POLITICS", "pilot400_rows": 4, "suggested_revision": "Làm rõ bài về chính sách kinh tế/tài chính: nếu trọng tâm là tác động thị trường, doanh nghiệp, ngân hàng, giá cả thì ECONOMY; nếu trọng tâm là quyết định/chủ thể quản trị nhà nước thì POLITICS."},
    ]

    report_parts = [
        "# IAA Analysis Technical Report",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "## Mục tiêu",
        "Phân tích mức nhất quán gán nhãn topic của 3 annotator (Nhung, Yen, Han) cho 2 vòng pilot: set 100, set 400, và 4 batch 100 dòng trong set 400.",
        "## Phương pháp",
        "- Fleiss' kappa được tính trên các dòng đủ cả 3 nhãn annotator.\n- Cohen's kappa được tính từng cặp annotator trên các dòng mà cả hai người trong cặp đều có nhãn.\n- `full_agreement` nghĩa là 3 người gán cùng một nhãn; `partial_disagreement` nghĩa là 2 người cùng nhãn và 1 người khác; `complete_disagreement` nghĩa là 3 người gán 3 nhãn khác nhau; `incomplete` nghĩa là thiếu ít nhất 1 nhãn annotator.\n- Universe nhãn gồm 18 topic labels xuất hiện trong toàn bộ pilot/final-train, dùng để xác định nhãn chưa được dùng ở từng tập.",
        "## Universe nhãn",
        md_table(label_rows, ["label_id", "label"]),
        "## Tổng quan chỉ số",
        md_table(overview, list(overview.columns)),
        "## Tổng quan phân phối và bất đồng trong 500 dòng",
        "Gộp pilot set 100 và pilot set 400 thành 500 dòng không trùng `record_id`. Phân phối cấp dòng dùng majority vote trên các dòng đủ 3 annotator; các dòng thiếu nhãn được ghi riêng là `INCOMPLETE_MISSING_LABELS`.",
        "### Phân phối nhãn cấp dòng trong 500 dòng",
        md_table(combined_majority_distribution, list(combined_majority_distribution.columns) if not combined_majority_distribution.empty else ["label", "rows", "pct_valid_rows", "pct_all_500_rows"]),
        "### Phân phối nhãn theo lượt gán trong 500 dòng",
        md_table(combined_assignment_distribution, list(combined_assignment_distribution.columns) if not combined_assignment_distribution.empty else ["label", "total_assignments", "Nhung", "Yen", "Han", "assignment_pct"]),
        "### Nhãn phát sinh bất đồng trong 500 dòng",
        md_table(combined_disagreement_labels, list(combined_disagreement_labels.columns) if not combined_disagreement_labels.empty else ["label_id", "label", "rows_with_label", "disagreement_rows_with_label", "disagreement_rate_when_label_appears", "disagreement_assignment_count"]),
        "### Cặp nhãn bất đồng trong 500 dòng (top 20)",
        md_table(combined_disagreement_pairs, list(combined_disagreement_pairs.columns) if not combined_disagreement_pairs.empty else ["label_a", "label_b", "disagreement_rows", "pct_of_disagreement_rows"]),
        "## Insight chính",
        "\n".join(insight_lines),
        "## Annotator giống trung bình nhóm / khác biệt so với nhóm",
        "\n".join(annotator_method_lines),
        md_table(annotator_overview, list(annotator_overview.columns)),
        "## Gợi ý những chỉnh sửa cho phiên bản guideline tiếp theo",
        "\n".join(guideline_lines),
        "### Gợi ý các ranh giới nên làm rõ",
        md_table(guideline_boundary_rows, ["suggested_boundary", "pilot400_rows", "suggested_revision"]),
    ]
    report_parts += [result_to_markdown(result) for result in results_full]
    report_parts.append("## Pilot 400 theo từng batch 100 dòng")
    report_parts += [result_to_markdown(result, top_n=10) for result in batch_results]
    return "\n\n".join(report_parts) + "\n"

# Load data and label universe
pilot100_df = read_pilot(PILOT100_PATH)
pilot400_df = read_pilot(PILOT400_PATH)
LABEL_UNIVERSE, LABEL_TO_ID = build_label_universe(pilot100_df, pilot400_df)

print(f"Repo root: {REPO_ROOT}")
print(f"Pilot 100: {PILOT100_PATH.name} -> {pilot100_df.shape[0]} rows, {pilot100_df.shape[1]} columns")
print(f"Pilot 400: {PILOT400_PATH.name} -> {pilot400_df.shape[0]} rows, {pilot400_df.shape[1]} columns")
print(f"Universe nhãn: {len(LABEL_UNIVERSE)} labels")
print(", ".join(LABEL_UNIVERSE))

In [ ]:
# 1. Phân tích file 100 dòng đầu
result_100 = analyze_dataset(
    name="Pilot set 100",
    df=pilot100_df,
    label_universe=LABEL_UNIVERSE,
    label_to_id=LABEL_TO_ID,
    row_scope="Toàn bộ file set 100 (100 dòng)",
)
print_analysis(result_100)

In [ ]:
# 2. Phân tích file 400 dòng
result_400 = analyze_dataset(
    name="Pilot set 400",
    df=pilot400_df,
    label_universe=LABEL_UNIVERSE,
    label_to_id=LABEL_TO_ID,
    row_scope="Toàn bộ file set 400 (400 dòng)",
)
print_analysis(result_400)

In [ ]:
# 3. Phân tích file 400 theo từng batch 100 dòng
batch_results = []
for batch_idx, start in enumerate(range(0, len(pilot400_df), 100), start=1):
    end = min(start + 100, len(pilot400_df))
    batch_df = pilot400_df.iloc[start:end].copy()
    result = analyze_dataset(
        name=f"Pilot set 400 - Batch {batch_idx}",
        df=batch_df,
        label_universe=LABEL_UNIVERSE,
        label_to_id=LABEL_TO_ID,
        row_scope=f"Dòng {start + 1}-{end} trong file set 400",
    )
    batch_results.append(result)
    print_analysis(result, top_n=8)

print("\nTổng quan batch")
batch_overview = overview_table(batch_results)
print(md_table(batch_overview, list(batch_overview.columns)))

In [ ]:
# Xuất technical report markdown
combined_500_df = pd.concat([pilot100_df, pilot400_df], ignore_index=True)
result_500 = analyze_dataset(
    name="Pilot set 100 + 400 (500 rows)",
    df=combined_500_df,
    label_universe=LABEL_UNIVERSE,
    label_to_id=LABEL_TO_ID,
    row_scope="Gộp pilot set 100 và pilot set 400 (500 dòng, record_id không trùng)",
)
REPORT_MD = build_report([result_100, result_400], batch_results, combined_result=result_500)
REPORT_PATH.write_text(REPORT_MD, encoding="utf-8")
print(f"Đã xuất report: {REPORT_PATH}")
print(f"Số ký tự report: {len(REPORT_MD):,}")